In [1]:
%pip install scikit-learn matplotlib pandas numpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

Note: you may need to restart the kernel to use updated packages.


In [2]:
features = ["PT1", "PT2", "P1", "P2", "TotalPT", "VertexChisq", "Isolation", "MASS"]

#load data
signal = pd.read_csv("data/signal_Bs2MuMu.txt", sep=r"\s+", header=None, names=features)
background = pd.read_csv("data/background_combinatorial.txt", sep=r"\s+", header=None, names=features)
signal = signal.apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)
background = background.apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)

#load the same top3 features computed in selection.ipynb
fisher_df = pd.read_csv("fisher_scores.csv")
top3_features = fisher_df.loc[fisher_df["Feature"] != "MASS", "Feature"].head(3).tolist()
print("top three features:", top3_features)

#load rectangular cut accuracy for comparison
with open("cut_params.json") as fh:
    cut_params = json.load(fh)
acc_rect = cut_params["accuracy"]
print(f"rectangular cut accuracy: {acc_rect:.4f}")

top three features: ['VertexChisq', 'Isolation', 'P2']
rectangular cut accuracy: 0.7500


(d)

In [3]:
#combine signal and background into a labelled dataset
def make_dataset(sig_df, bkg_df, feat_list):
    X = pd.concat([sig_df[feat_list], bkg_df[feat_list]], axis=0).reset_index(drop=True)
    y = np.concatenate([np.ones(len(sig_df)), np.zeros(len(bkg_df))])
    return X, y

X3, y3 = make_dataset(signal, background, top3_features)

#split into train and test - evaluate only on unseen data to avoid overfitting bias
X_train, X_test, y_train, y_test = train_test_split(
    X3, y3, test_size=0.3, random_state=42, stratify=y3
)
print("training size:", len(X_train), "  test size:", len(X_test))

training size: 14000   test size: 6000


In [4]:
#keep base trees shallow to prevent overfitting
base_tree = DecisionTreeClassifier(max_depth=2, min_samples_leaf=50, random_state=42)

bdt3 = AdaBoostClassifier(
    estimator=base_tree, n_estimators=100, learning_rate=0.5, random_state=42
)
bdt3.fit(X_train, y_train)

y_pred3 = bdt3.predict(X_test)
acc3    = accuracy_score(y_test, y_pred3)
cm3     = confusion_matrix(y_test, y_pred3)
tn, fp, fn, tp = cm3.ravel()

print(f"BDT with top 3 features: {top3_features}")
print(f"accuracy              = {acc3:.4f}")
print(f"signal efficiency     = {tp/(tp+fn):.4f}")
print(f"background efficiency = {fp/(fp+tn):.4f}")
print(f"background rejection  = {tn/(tn+fp):.4f}")
print(f"\nconfusion matrix [[TN, FP], [FN, TP]]:\n{cm3}")
print(f"\nrectangular cut accuracy: {acc_rect:.4f}  ->  BDT improvement: {acc3 - acc_rect:.4f}")

BDT with top 3 features: ['VertexChisq', 'Isolation', 'P2']
accuracy              = 0.8640
signal efficiency     = 0.8623
background efficiency = 0.1343
background rejection  = 0.8657

confusion matrix [[TN, FP], [FN, TP]]:
[[2597  403]
 [ 413 2587]]

rectangular cut accuracy: 0.7500  ->  BDT improvement: 0.1140


(e)

In [5]:
#now use all 7 features (still excluding MASS)
features7 = [f for f in features if f != "MASS"]
print("using all 7 features:", features7)

X7, y7 = make_dataset(signal, background, features7)
X_train7, X_test7, y_train7, y_test7 = train_test_split(
    X7, y7, test_size=0.3, random_state=42, stratify=y7
)

bdt7 = AdaBoostClassifier(
    estimator=base_tree, n_estimators=100, learning_rate=0.5, random_state=42
)
bdt7.fit(X_train7, y_train7)

y_pred7  = bdt7.predict(X_test7)
acc7     = accuracy_score(y_test7, y_pred7)
cm7      = confusion_matrix(y_test7, y_pred7)
tn7, fp7, fn7, tp7 = cm7.ravel()

print(f"accuracy              = {acc7:.4f}")
print(f"signal efficiency     = {tp7/(tp7+fn7):.4f}")
print(f"background efficiency = {fp7/(fp7+tn7):.4f}")
print(f"background rejection  = {tn7/(tn7+fp7):.4f}")
print(f"\nconfusion matrix [[TN, FP], [FN, TP]]:\n{cm7}")
print(f"\naccuracy 3 features = {acc3:.4f}  ->  7 features = {acc7:.4f}  (improvement: {acc7 - acc3:.4f})")

using all 7 features: ['PT1', 'PT2', 'P1', 'P2', 'TotalPT', 'VertexChisq', 'Isolation']
accuracy              = 0.9287
signal efficiency     = 0.9313
background efficiency = 0.0740
background rejection  = 0.9260

confusion matrix [[TN, FP], [FN, TP]]:
[[2778  222]
 [ 206 2794]]

accuracy 3 features = 0.8640  ->  7 features = 0.9287  (improvement: 0.0647)


In [6]:
#feature importances from the 7-feature BDT
importance_df = pd.DataFrame({
    "Feature":    features7,
    "Importance": bdt7.feature_importances_
}).sort_values("Importance", ascending=False).reset_index(drop=True)

display(importance_df)

,Feature,Importance
0,TotalPT,0.234009
1,P2,0.168795
2,VertexChisq,0.162163
3,P1,0.147471
4,Isolation,0.113016
5,PT2,0.095439
6,PT1,0.079106


(f)

In [7]:
#apply the 7-feature BDT to the full dataset and count how many events pass
sig_pass  = bdt7.predict(signal[features7]).astype(bool)
bkg_pass  = bdt7.predict(background[features7]).astype(bool)

n_sig_pass = int(sig_pass.sum())
n_bkg_pass = int(bkg_pass.sum())
sig_eff    = n_sig_pass / len(signal)
bkg_eff    = n_bkg_pass / len(background)

print("events passing BDT selection:")
print(f"  signal:     {n_sig_pass} / {len(signal)}   (efficiency = {sig_eff:.4f})")
print(f"  background: {n_bkg_pass} / {len(background)}   (efficiency = {bkg_eff:.4f})")

events passing BDT selection:
  signal:     9305 / 10000   (efficiency = 0.9305)
  background: 726 / 10000   (efficiency = 0.0726)


In [8]:
#save the BDT model and efficiencies for use in mass_fit.ipynb
with open("bdt_model.pkl", "wb") as fh:
    pickle.dump(bdt7, fh)

results = {
    "acc_rect":               acc_rect,
    "acc_3feat":              float(acc3),
    "acc_7feat":              float(acc7),
    "signal_efficiency":      float(sig_eff),
    "background_efficiency":  float(bkg_eff),
    "n_sig_pass":             n_sig_pass,
    "n_bkg_pass":             n_bkg_pass,
}
with open("bdt_results.json", "w") as fh:
    json.dump(results, fh, indent=2)

print("saved bdt_model.pkl and bdt_results.json")

saved bdt_model.pkl and bdt_results.json
